In [95]:
import pandas as pd
import numpy as np


In [96]:
patients = {
    "patient_id": ["P001","P002","P003","P004","P005","P006","P007","P008","P009","P010"],
    "age": [65, 45, 72, 30, 58, 51, 75, 67, 40, 80],
    "gender": ["M","F","M","F","M","F","M","F","M","F"],
    "diagnosis": ["HTN","DM","COPD","Asthma","HTN","Stroke","DM","COPD","Asthma","DM"],
    "diabetic": [True, False, True, False, False, False, True, False, False, True],
    "sbp": [140, 120, 150, 110, 130, 79, 145, 125, 118, 81],
    "spo2": [92, 98, 88, 97, 95, 90, 93, 96, 99, 89],
    "glucose_level": [210, 140, 190, 110, 250, 160, 175, 220, 113, 202],
    "icu_admission": ["yes","no","yes","no","yes","yes","no","yes","no","yes"]
}

df = pd.DataFrame(patients)
df

,patient_id,age,gender,diagnosis,diabetic,sbp,spo2,glucose_level,icu_admission
0,P001,65,M,HTN,True,140,92,210,yes
1,P002,45,F,DM,False,120,98,140,no
2,P003,72,M,COPD,True,150,88,190,yes
3,P004,30,F,Asthma,False,110,97,110,no
4,P005,58,M,HTN,False,130,95,250,yes
5,P006,51,F,Stroke,False,79,90,160,yes
6,P007,75,M,DM,True,145,93,175,no
7,P008,67,F,COPD,False,125,96,220,yes
8,P009,40,M,Asthma,False,118,99,113,no
9,P010,80,F,DM,True,81,89,202,yes


In [97]:
df["risk_flag"] = (df["age"] >= 60) | (df["spo2"] < 92)

df["severity_score"] = 0

df.loc[df["age"] > 65, "severity_score"] += 2
df.loc[df["diabetic"], "severity_score"] += 1
df.loc[df["spo2"] < 92, "severity_score"] += 3

df

,patient_id,age,gender,diagnosis,diabetic,sbp,spo2,glucose_level,icu_admission,risk_flag,severity_score
0,P001,65,M,HTN,True,140,92,210,yes,True,1
1,P002,45,F,DM,False,120,98,140,no,False,0
2,P003,72,M,COPD,True,150,88,190,yes,True,6
3,P004,30,F,Asthma,False,110,97,110,no,False,0
4,P005,58,M,HTN,False,130,95,250,yes,False,0
5,P006,51,F,Stroke,False,79,90,160,yes,True,3
6,P007,75,M,DM,True,145,93,175,no,True,3
7,P008,67,F,COPD,False,125,96,220,yes,True,2
8,P009,40,M,Asthma,False,118,99,113,no,False,0
9,P010,80,F,DM,True,81,89,202,yes,True,6


In [98]:
# 1) What is the average SpO2 in diabetic vs non-diabetic patients?
df.groupby("diabetic")["spo2"].mean()

diabetic
False    95.833333
True     90.500000
Name: spo2, dtype: float64

In [114]:
# 2) Which diagnosis has the highest average SBP?
df.groupby("diagnosis")["sbp"].mean().sort_values(ascending=False).head(1)
# df.groupby("diagnosis")["sbp"].mean().idxmax()



diagnosis
COPD    137.5
Name: sbp, dtype: float64

In [ ]:
# 3) How many patients are at high risk by your risk_flag definition?
df["risk_flag"].sum()

np.int64(6)

In [116]:
# 4) What percentage of ICU patients are diabetic?
# icu = df[df["icu_admission"] == "yes"]
# diabetic = icu[icu["diabetic"]]
# percentage = (len(diabetic) / len(icu)) * 100
# print(percentage)

df[df["icu_admission"] == "yes"]["diabetic"].mean() * 100

# percentage = (df[df["icu_admission"] == "yes"]["diabetic"].mean()) * 100
# print(percentage)

np.float64(50.0)

In [102]:
# 5) Who are the top 3 most severe patients by severity_score?
df.sort_values("severity_score", ascending= False).head(3)

,patient_id,age,gender,diagnosis,diabetic,sbp,spo2,glucose_level,icu_admission,risk_flag,severity_score
9,P010,80,F,DM,True,81,89,202,yes,True,6
2,P003,72,M,COPD,True,150,88,190,yes,True,6
6,P007,75,M,DM,True,145,93,175,no,True,3


In [119]:
# 6) What is the gender distribution in ICU admissions?
# df[df["icu_admission"] == "yes"].groupby("gender").size()
df[df["icu_admission"] == "yes"]["gender"].value_counts()

gender
M    3
F    3
Name: count, dtype: int64

In [104]:
# 7) What is the average glucose in patients with sbp < 90?
df[df["sbp"] < 90]["glucose_level"].mean()

np.float64(181.0)

In [120]:
# 8) How does severity_category distribute across the whole ward?
df["severity_category"] = ""

df.loc[df["severity_score"] <= 2, "severity_category"] = "Low"
df.loc[(df["severity_score"] >= 3) & (df["severity_score"] <= 4), "severity_category"] = "Moderate"
df.loc[df["severity_score"] >= 5, "severity_category"] = "High"

df["severity_category"].value_counts()

severity_category
Low         6
High        2
Moderate    2
Name: count, dtype: int64